> https://github.com/nmcassa/letterboxdpy

> https://letterboxd.com/


> **Sobre a biblioteca:** `Movie(slug).get_reviews()`
>  é um stub incompleto na versão atual, sempre retorna vazio. O que
**funciona de
> verdade** é `Movie(slug).popular_reviews`, preenchido automaticamente ao instanciar `Movie`: ele lê a
> seção "Popular Reviews" já presente na página do filme. Duas limitações a ter em mente:
> - **sem paginação** — só traz as resenhas exibidas naquela seção da página (não dá pra pedir mais páginas);
> - **só o primeiro parágrafo** de cada resenha é capturado pelo parser da biblioteca.
>
> Por isso, para ganhar volume e variedade (critério do template), a coleta varia os **filmes**
> (populares + vários gêneros) em vez de tentar paginar resenhas de um único filme.


In [ ]:
!pip install -q git+https://github.com/nmcassa/letterboxdpy.git

!pip install -q nltk langdetect unidecode pandas tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.7 MB/s eta 0:00:00


In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from langdetect import detect, DetectorFactory, LangDetectException

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer, SnowballStemmer

from letterboxdpy.films import Films, get_movies_by_genre
from letterboxdpy.movie import Movie
from letterboxdpy.core.exceptions import (
    PrivateRouteError,
    ResourceNotFoundError,
    AccessDeniedError,
    InvalidResponseError,
    PageLoadError,
)

DetectorFactory.seed = 42


In [ ]:
GENEROS = [
    "action", "comedy", "drama", "horror", "romance",
    "science-fiction", "documentary", "animation", "adventure",
    "crime", "family", "fantasy", "thriller", "history", "music",
    "tv-movie", "war", "western", "musical"
]
MAX_POR_LISTA = 10       # filmes por lista (populares + cada gênero)
PAUSA_ENTRE_REQS = 1.0   # segundos entre requisições, para não sobrecarregar o servidor
ARQ_BRUTO = Path("resenhas_letterboxd_bruto.csv")

print("Buscando filmes populares...")
filmes = dict(Films("https://letterboxd.com/films/popular/", max=MAX_POR_LISTA).movies)

for genero in GENEROS:
    print(f"Buscando filmes do gênero '{genero}'...")
    filmes |= get_movies_by_genre(genero, max=MAX_POR_LISTA)
    time.sleep(PAUSA_ENTRE_REQS)

slugs = sorted({info["slug"] for info in filmes.values()})
print(f"\n{len(slugs)} filmes únicos reunidos (populares + {len(GENEROS)} gêneros).")


Buscando filmes populares...
Buscando filmes do gênero 'action'...
Buscando filmes do gênero 'comedy'...
Buscando filmes do gênero 'drama'...
Buscando filmes do gênero 'horror'...
Buscando filmes do gênero 'romance'...
Buscando filmes do gênero 'science-fiction'...
Buscando filmes do gênero 'documentary'...
Buscando filmes do gênero 'animation'...

61 filmes únicos reunidos (populares + 8 gêneros).


In [ ]:
def coletar_resenhas_por_filme(slugs, pausa=PAUSA_ENTRE_REQS):
    registros = []
    erros = []

    for slug in tqdm(slugs, desc="Coletando por filme"):
        try:
            m = Movie(slug)
        except (PrivateRouteError, ResourceNotFoundError, AccessDeniedError,
                InvalidResponseError, PageLoadError) as e:
            erros.append({"filme_slug": slug, "erro": str(type(e).__name__)})
            continue
        except Exception as e:
            erros.append({"filme_slug": slug, "erro": f"inesperado: {e}"})
            continue

        generos = [g["name"] for g in (m.genres or [])]
        diretores = [d["name"] for d in m.crew.get("director", [])] if m.crew else []

        for rev in (m.popular_reviews or []):
            texto = rev.get("review")
            if not texto:
                continue
            registros.append({
                "filme": m.title,
                "filme_slug": slug,
                "ano_lancamento": m.year,
                "generos": generos,
                "diretores": diretores,
                "nota_media_letterboxd": m.rating,
                "usuario": (rev.get("user") or {}).get("username"),
                "nota_resenha": rev.get("rating"),
                "link": rev.get("link"),
                "resenha_original": texto,
            })

        time.sleep(pausa)

    return registros, erros

registros, erros_coleta = coletar_resenhas_por_filme(slugs)
print(f"\n{len(registros)} resenhas coletadas | {len(erros_coleta)} filmes pulados por erro.")


Coletando por filme:   0%|          | 0/61 [00:00<?, ?it/s]


732 resenhas coletadas | 0 filmes pulados por erro.


In [ ]:
df = pd.DataFrame(registros)
df.drop_duplicates(subset=["filme_slug", "usuario", "resenha_original"], inplace=True)
df.to_csv(ARQ_BRUTO, index=False, encoding="utf-8")
print(f"Base bruta salva em: {ARQ_BRUTO.resolve()}")
df.head()


Base bruta salva em: /content/resenhas_letterboxd_bruto.csv


,filme,filme_slug,ano_lancamento,generos,diretores,nota_media_letterboxd,usuario,nota_resenha,link,resenha_original
0,10 Things I Hate About You,10-things-i-hate-about-you,1999,"[Comedy, Drama, Romance, Underdogs and coming ...",[Gil Junger],4.07,riverjphoenix,None,https://letterboxd.com/riverjphoenix/film/10-t...,the moral of the story is that all men are evi...
1,10 Things I Hate About You,10-things-i-hate-about-you,1999,"[Comedy, Drama, Romance, Underdogs and coming ...",[Gil Junger],4.07,riverjphoenix,None,https://letterboxd.com/riverjphoenix/film/10-t...,kat stratford is the most relatable character ...
2,10 Things I Hate About You,10-things-i-hate-about-you,1999,"[Comedy, Drama, Romance, Underdogs and coming ...",[Gil Junger],4.07,vortexd,None,https://letterboxd.com/vortexd/film/10-things-...,me: i hate cliches
3,10 Things I Hate About You,10-things-i-hate-about-you,1999,"[Comedy, Drama, Romance, Underdogs and coming ...",[Gil Junger],4.07,sapphical,None,https://letterboxd.com/sapphical/film/10-thing...,funny how shakespeare’s been real quiet since ...
4,10 Things I Hate About You,10-things-i-hate-about-you,1999,"[Comedy, Drama, Romance, Underdogs and coming ...",[Gil Junger],4.07,jay,None,https://letterboxd.com/jay/film/10-things-i-ha...,heath ledger: *dances down steps with giant le...


In [ ]:
STOPWORDS_PT = set(stopwords.words("portuguese"))
STOPWORDS_EN = set(stopwords.words("english"))

stemmer_pt = RSLPStemmer()
stemmer_en = SnowballStemmer("english")

def limpar_ruido(texto):
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def detectar_idioma(texto):
    try:
        return detect(texto)
    except LangDetectException:
        return "desconhecido"

def tokenizar_e_limpar(texto, idioma):
    texto = texto.lower()
    tokens = word_tokenize(texto)
    tokens = [t for t in tokens if t.isalpha()]

    if idioma == "pt":
        stop = STOPWORDS_PT
    elif idioma == "en":
        stop = STOPWORDS_EN
    else:
        stop = set()

    return [t for t in tokens if t not in stop]

def aplicar_stemming(tokens, idioma):
    if idioma == "pt":
        return [stemmer_pt.stem(t) for t in tokens]
    if idioma == "en":
        return [stemmer_en.stem(t) for t in tokens]
    return tokens


In [ ]:
tqdm.pandas()

df["resenha_limpa"] = df["resenha_original"].progress_apply(limpar_ruido)
df = df[df["resenha_limpa"].str.len() > 0].copy()

df["idioma"] = df["resenha_limpa"].progress_apply(detectar_idioma)
df["tokens"] = df.progress_apply(lambda r: tokenizar_e_limpar(r["resenha_limpa"], r["idioma"]), axis=1)
df["tokens_stemizados"] = df.progress_apply(lambda r: aplicar_stemming(r["tokens"], r["idioma"]), axis=1)

print("Distribuição de idiomas detectados:")
print(df["idioma"].value_counts())
df[["filme", "resenha_original", "idioma", "tokens"]].head()


  0%|          | 0/732 [00:00<?, ?it/s]

  0%|          | 0/732 [00:00<?, ?it/s]

  0%|          | 0/732 [00:00<?, ?it/s]

  0%|          | 0/732 [00:00<?, ?it/s]

Distribuição de idiomas detectados:
idioma
en              667
cy                7
nl                6
desconhecido      6
de                5
af                4
no                4
it                4
fr                3
tl                3
sv                3
da                3
so                2
id                2
hu                2
sk                2
fi                1
sw                1
es                1
ro                1
sq                1
vi                1
tr                1
et                1
hr                1
Name: count, dtype: int64


,filme,resenha_original,idioma,tokens
0,10 Things I Hate About You,the moral of the story is that all men are evi...,en,"[moral, story, men, evil, except, heath, ledge..."
1,10 Things I Hate About You,kat stratford is the most relatable character ...,en,"[kat, stratford, relatable, character, time, d..."
2,10 Things I Hate About You,me: i hate cliches,en,"[hate, cliches]"
3,10 Things I Hate About You,funny how shakespeare’s been real quiet since ...,en,"[funny, shakespeare, real, quiet, since, josep..."
4,10 Things I Hate About You,heath ledger: *dances down steps with giant le...,en,"[heath, ledger, dances, steps, giant, leaps]"


## Dicionário de dados


| Coluna | Tipo | Descrição |
|---|---|---|
| `filme` | str | Título do filme |
| `filme_slug` | str | Slug do filme no Letterboxd |
| `ano_lancamento` | int | Ano de lançamento do filme |
| `generos` | list[str] | Gêneros do filme |
| `diretores` | list[str] | Diretor(es) do filme |
| `nota_media_letterboxd` | float | Nota média do filme na plataforma |
| `usuario` | str | Nome de usuário de quem escreveu a resenha |
| `nota_resenha` | float | Nota dada pelo usuário naquela resenha (0.5 a 5.0) |
| `link` | str | URL da resenha |
| `resenha_original` | str | Texto bruto da resenha, como coletado (primeiro parágrafo) |
| `resenha_limpa` | str | Texto após remoção de ruído (HTML, URLs, espaços) |
| `idioma` | str | Idioma detectado (`pt`, `en`, outro código ISO ou `desconhecido`) |
| `tokens` | list[str] | Tokens após normalização, tokenização e remoção de stopwords |
| `tokens_stemizados` | list[str] | Tokens após stemming (bonus) |


In [ ]:
ARQ_FINAL = Path("resenhas_letterboxd_processado.csv")

df_export = df.copy()
df_export["tokens"] = df_export["tokens"].apply(json.dumps, ensure_ascii=False)
df_export["tokens_stemizados"] = df_export["tokens_stemizados"].apply(json.dumps, ensure_ascii=False)
df_export["generos"] = df_export["generos"].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)
df_export["diretores"] = df_export["diretores"].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)

df_export.to_csv(ARQ_FINAL, index=False, encoding="utf-8")
print(f"Base final salva em: {ARQ_FINAL.resolve()}")
print(f"Total de resenhas no dataset final: {len(df_export)}")
print(f"Total de filmes distintos: {df_export['filme_slug'].nunique()}")


Base final salva em: /content/resenhas_letterboxd_processado.csv
Total de resenhas no dataset final: 732
Total de filmes distintos: 61
